<p style="text-align: center; margin-bottom: 0; font-size: 24px;">EAE1106 - Métodos Computacionais para Economia</p>
<p style="text-align: center; margin-bottom: 0;">Faculdade de Economia, Administração, Contabilidade e Atuária<br>Universidade de São Paulo</p>
<p style="text-align: center; margin-top: 5px;">1º semestre de 2026<br><br>Prof. Arthur Viaro</p>

# Aula 23 — Seaborn e Plotly

**Conteúdo**

- [O que são Seaborn e Plotly?](#intro)
- [Seaborn: visualizações estatísticas](#seaborn)
  - [Primeiro gráfico](#primeiro-grafico)
  - [Funções axes-level e figure-level](#api)
  - [Gráfico de linha](#lineplot)
  - [Gráfico de dispersão](#scatterplot)
  - [Distribuições: histograma e KDE](#distribuicoes)
  - [Boxplot](#boxplot)
  - [Mapa de calor](#heatmap)
  - [Temas e paletas](#temas)
- [Plotly: gráficos interativos](#plotly)
  - [Gráfico de linha interativo](#plotly-linha)
  - [Múltiplos países](#plotly-multi)
  - [Dispersão interativa](#plotly-scatter)
  - [Barras interativas](#plotly-bar)
- [Quando usar cada biblioteca?](#comparacao)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

## O que são Seaborn e Plotly? <a id="intro"></a>

Na aula anterior, aprendemos a criar e personalizar gráficos com o **Matplotlib** — a biblioteca base de visualização em Python. Hoje veremos duas bibliotecas que ampliam esse ecossistema:

- **Seaborn** é construído diretamente sobre o Matplotlib e foi projetado para visualizações estatísticas. Com menos linhas de código, produz gráficos mais elegantes e é especialmente útil para explorar distribuições e relações entre variáveis. Como um gráfico Seaborn *é* um gráfico Matplotlib por baixo dos panos, todas as personalizações da aula anterior continuam funcionando.

- **Plotly** segue uma filosofia diferente: seus gráficos são **interativos** — é possível dar zoom, passar o mouse sobre os pontos para ver informações, e ativar/desativar séries. Isso o torna especialmente útil para exploração de dados e apresentações em formato web ou HTML.

Como na aula anterior, os exemplos usam os dados da **Penn World Table 11.0**:

| Variável | Descrição |
|----------|-----------|
| `country` | Nome do país |
| `year` | Ano |
| `pop` | População (milhões) |
| `emp` | Pessoas empregadas (milhões) |
| `avh` | Horas médias trabalhadas por trabalhador no ano |
| `rgdpna` | PIB real (milhões de USD de 2017) |

In [ ]:
columns_to_read = ['country', 'year', 'pop', 'emp', 'rgdpna', 'avh']
df_pwt = pd.read_csv('pwt110.csv', usecols=columns_to_read)
df_pwt['pibpc'] = df_pwt['rgdpna'] / df_pwt['pop']
df_pwt.info()

## Seaborn: visualizações estatísticas <a id="seaborn"></a>

O Seaborn é importado com a abreviação convencional `sns`. A filosofia central é uma API **declarativa**: em vez de especificar cores, marcadores e formatos linha a linha, indicamos de qual coluna do DataFrame vem cada elemento do gráfico — e o Seaborn cuida do restante.

Vamos aprender o Seaborn de forma progressiva, começando pelo gráfico mais simples possível e adicionando recursos aos poucos.


### Primeiro gráfico <a id="primeiro-grafico"></a>

Com apenas três linhas de código, o Seaborn já produz um gráfico mais elegante que o Matplotlib padrão. A chamada `sns.set_theme()` aplica um tema visual agradável globalmente, antes de qualquer gráfico ser criado:


In [ ]:
df_brazil = df_pwt[df_pwt['country'] == 'Brazil']

sns.set_theme()
sns.lineplot(data=df_brazil, x='year', y='pibpc')

Repare no que aconteceu com apenas três linhas:

- `sns.set_theme()` aplicou o tema padrão (`'darkgrid'`) — fundo cinza com grade — que já melhora a estética automaticamente.
- `data=df_brazil` passa o DataFrame inteiro para o Seaborn.
- `x='year'` e `y='pibpc'` indicam as colunas como **strings** — os eixos são rotulados automaticamente.

Nenhuma instrução de cor, marcador ou estilo foi necessária.

### Funções axes-level e figure-level <a id="api"></a>

O Seaborn divide suas funções em dois grupos, e entender essa distinção é fundamental para escrever código limpo.

**Funções axes-level** plotam dados em um único objeto `Axes` do Matplotlib e retornam esse objeto. São funções como `sns.lineplot()`, `sns.scatterplot()`, `sns.histplot()`, `sns.kdeplot()` e `sns.boxplot()`. Usam-se quando você já tem uma figura criada com `plt.subplots()` e quer inserir o gráfico em um eixo específico — por isso aceitam o parâmetro `ax=`.

**Funções figure-level** gerenciam toda a figura por conta própria, através de um objeto `FacetGrid`. Elas não precisam de `plt.subplots()` e controlam tamanho, bordas e rótulos com métodos próprios. Cada módulo temático do Seaborn tem exatamente uma função figure-level:

| Função figure-level | Funções axes-level equivalentes |
|---|---|
| `relplot()` | `lineplot()`, `scatterplot()` |
| `displot()` | `histplot()`, `kdeplot()`, `ecdfplot()` |
| `catplot()` | `boxplot()`, `violinplot()`, `stripplot()`, `barplot()`, ... |

A diferença prática está na forma de customizar o resultado:

| Operação | Axes-level | Figure-level |
|----------|-----------|--------------|
| Tamanho da figura | `figsize=` em `plt.subplots()` | `height=` e `aspect=` |
| Rótulos dos eixos | `ax.set_xlabel()` / `ax.set_ylabel()` | `g.set_axis_labels()` |
| Título | `ax.set_title()` | `g.figure.suptitle()` |
| Remover bordas | `ax.spines[...].set_visible(False)` | `g.despine()` |
| Título da legenda | `ax.legend(title=...)` | `g._legend.set_title()` |

Nas próximas seções, usaremos **sempre as funções figure-level** para não depender do Matplotlib diretamente. A única exceção é o `heatmap`, que não tem equivalente figure-level — mas mesmo assim não precisamos de `plt.subplots()`, pois o Seaborn cria a figura internamente.


### Gráfico de linha <a id="lineplot"></a>

Para ter mais controle sobre título, rótulos e tamanho, usamos a função de **nível figura** `relplot()`. Ela retorna um objeto `g` com métodos próprios — sem precisar de `plt.subplots()` ou `ax`:

In [ ]:
df_pwtbr = df_pwt.loc[df_pwt['country'] == 'Brazil'].copy()
df_pwtbr['avh_dia'] = df_pwtbr['avh'] / 260

sns.set_theme(style='white')

g = sns.relplot(
    data=df_pwtbr,
    x='year',
    y='avh_dia',
    kind='line',
    height=5,
    aspect=1.6
)

g.set_axis_labels('Ano', 'Horas por dia útil')
g.figure.suptitle('Horas médias trabalhadas por dia útil — Brasil', fontsize=13, y=1.02)
g.despine()

O objeto retornado `g` já tem métodos integrados: `g.set_axis_labels()` define os rótulos dos eixos, `g.figure.suptitle()` adiciona o título e `g.despine()` remove as bordas — tudo sem escrever uma linha de código Matplotlib.

Uma das principais vantagens do Seaborn aparece quando queremos comparar **múltiplos grupos**: basta usar o argumento `hue=` com o nome da coluna que define os grupos, e o Seaborn plota cada grupo em uma cor diferente e gera a legenda automaticamente:


In [ ]:
countries = ['Brazil', 'United States', 'Germany', 'Japan', 'China']
df_multi = df_pwt[df_pwt['country'].isin(countries)].copy()
df_multi['avh_dia'] = df_multi['avh'] / 260
df_multi['country'] = df_multi['country'].replace({'United States': 'EUA'})

sns.set_theme(style='whitegrid')

g = sns.relplot(
    data=df_multi,
    x='year',
    y='avh_dia',
    hue='country',
    kind='line',
    linewidth=1.8,
    height=6,
    aspect=1.7
)

g.set_axis_labels('Ano', 'Horas por dia útil')
g.set_titles('')
g.ax.set_title('Horas médias trabalhadas por dia útil — países selecionados', fontsize=13)
g._legend.set_title('País')

sns.despine()

Com apenas o argumento `hue='country'`, o Seaborn plotou cinco países em cores distintas e criou a legenda automaticamente — o que exigiria um loop e muito mais código no Matplotlib puro.

### Gráfico de dispersão <a id="scatterplot"></a>

Na aula anterior criamos um gráfico de dispersão entre horas trabalhadas e PIB per capita. Com `relplot()` (o padrão, quando `kind=` não é especificado, é dispersão) podemos ir além: o argumento `hue=` adiciona uma **terceira dimensão de informação** diferenciando os pontos por cor.

Vamos classificar os países de 2010 em três grupos de renda com base no PIB per capita:


In [ ]:
df_2010 = df_pwt[df_pwt['year'] == 2010].dropna(subset=['avh', 'pibpc']).copy()
df_2010['log_pibpc'] = np.log(df_2010['pibpc'])

# Classificar países em três grupos de renda por tercis do PIB per capita
t33, t67 = df_2010['pibpc'].quantile([1/3, 2/3])

def grupo_renda(x):
    if x <= t33:
        return 'Baixa renda'
    elif x <= t67:
        return 'Renda média'
    else:
        return 'Alta renda'

df_2010['grupo'] = df_2010['pibpc'].apply(grupo_renda)
print(df_2010['grupo'].value_counts())

In [ ]:
sns.set_theme(style='white')

sns.scatterplot(
    data=df_2010,
    x='avh',
    y='log_pibpc',
    hue='grupo',
    palette='Set2', # deep, muted, bright, pastel, dark, colorblind
    s=50
)

In [ ]:
palette = {'Alta renda': '#2ca02c', 'Renda média': '#ff7f0e', 'Baixa renda': '#d62728'}

sns.set_theme(style='white')

g = sns.relplot(
    data=df_2010,
    x='avh',
    y='log_pibpc',
    hue='grupo',
    hue_order=['Alta renda', 'Renda média', 'Baixa renda'],
    kind='scatter',
    palette=palette,
    alpha=0.75,
    s=60,
    height=5,
    aspect=1.6
)

g.set_axis_labels('Horas médias trabalhadas no ano (avh)', 'Log do PIB per capita (USD 2017)')
g.figure.suptitle('Horas trabalhadas vs. PIB per capita — 2010', fontsize=13, y=1.02)
g._legend.set_title('Grupo de renda')
g.despine()

Os países de **alta renda** (verde) tendem a se concentrar na parte inferior do eixo X (menos horas trabalhadas) e superior do eixo Y (maior PIB per capita), evidenciando o padrão discutido na literatura sobre desenvolvimento econômico.

### Distribuições: histograma e KDE <a id="distribuicoes"></a>

A função de nível figura para distribuições é `sns.displot()`. O argumento `kind=` seleciona o tipo de visualização:

- `kind='hist'` (padrão): histograma, contando observações por intervalo
- `kind='kde'`: estimação por *kernel* de densidade — uma versão contínua e suavizada do histograma

#### Histograma


In [ ]:
sns.set_theme(style='white')

sns.histplot(data=df_2010, x="avh", bins=25)

In [ ]:
sns.set_theme(style='white')

g = sns.displot(
    data=df_2010,
    x='avh',
    kind='hist',  # hist, kde, ecdf (hist é o padrão)
    bins=25,
    color='steelblue',
    height=5,
    aspect=1.7
)

g.set_axis_labels('Horas médias trabalhadas no ano (avh)', 'Número de países')
g.figure.suptitle('Distribuição das horas trabalhadas entre países — 2010', fontsize=13, y=1.02)
g.despine()

O parâmetro `bins=` controla o número de intervalos. Um valor muito baixo cria um histograma grosseiro; muito alto, intervalos esparsos.

#### Estimador de densidade (KDE)

O `sns.kdeplot()` produz uma curva contínua que estima a distribuição dos dados. Uma das suas aplicações mais úteis é **sobrepor a distribuição de múltiplos grupos** no mesmo gráfico, facilitando a comparação:

In [ ]:
sns.set_theme(style='white')

sns.kdeplot(data=df_2010, x='avh', fill=True, color='steelblue')

In [ ]:
palette = {'Alta renda': '#2ca02c', 'Renda média': '#ff7f0e', 'Baixa renda': '#d62728'}

sns.set_theme(style='white')

g = sns.displot(
    data=df_2010,
    x='avh',
    hue='grupo',
    hue_order=['Alta renda', 'Renda média', 'Baixa renda'],
    kind='kde',  # hist, kde, ecdf
    palette=palette,
    linewidth=2,
    height=5,
    aspect=1.7
)

g.set_axis_labels('Horas médias trabalhadas no ano (avh)', 'Densidade')
g.figure.suptitle('Horas trabalhadas por grupo de renda — 2010', fontsize=13, y=1.02)
g._legend.set_title('Grupo de renda')
g.despine()

O KDE deixa evidente que a distribuição das horas trabalhadas difere entre grupos de renda: países ricos concentram-se em jornadas mais curtas.

### Boxplot <a id="boxplot"></a>

O **boxplot** (diagrama de caixa) é uma forma compacta de visualizar distribuições: mostra a mediana (linha central), os quartis (a caixa) e os valores atípicos (*outliers*, pontos isolados). Com `sns.catplot(kind='box')` comparamos distribuições entre grupos com uma única chamada.

Vamos ver como as horas trabalhadas no Brasil variaram por década — cada caixa resume os anos daquela década:


In [ ]:
df_pwtbr_dec = df_pwt[df_pwt['country'] == 'Brazil'].copy()

# Transformar o ano em década, transforma em texto e adiciona "s" no final
df_pwtbr_dec['decada'] = (df_pwtbr_dec['year'] // 10 * 10).astype(str) + 's'

# Converte horas trabalhadas por ano em horas trabalhadas por dia útil
df_pwtbr_dec['avh_dia'] = df_pwtbr_dec['avh'] / 260

In [ ]:
sns.set_theme(style='whitegrid')

sns.boxplot(
    data=df_pwtbr_dec,
    x='decada',
    y='avh_dia',
    color='steelblue'
)

In [ ]:
sns.set_theme(style='whitegrid')

g = sns.catplot(
    data=df_pwtbr_dec,
    x='decada',
    y='avh_dia',
    kind='box', # strip, swarm, box, violin, boxen, point, bar
    hue='decada',
    palette='Blues',
    legend=False,
    height=5,
    aspect=1.8
)

g.set_axis_labels('Década', 'Horas por dia útil')
g.figure.suptitle('Horas trabalhadas por dia útil no Brasil, por década', fontsize=12, y=1.02)
g.despine(left=True)

Cada caixa agrupa as observações anuais de uma década. A queda clara da mediana ao longo do tempo confirma a tendência de redução da jornada de trabalho no Brasil.

### Mapa de calor (Heatmap) <a id="heatmap"></a>

O `sns.heatmap()` é especialmente útil para visualizar **matrizes de correlação**: usa uma escala de cores para representar o coeficiente de correlação entre cada par de variáveis. O parâmetro `annot=True` exibe os valores numéricos dentro de cada célula:

In [ ]:
df_corr = df_pwt[df_pwt['year'] == 2010].dropna(subset=['pop', 'emp', 'avh', 'pibpc']).copy()
df_corr['log_pibpc'] = np.log(df_corr['pibpc'])

variaveis = df_corr[['pop', 'emp', 'avh', 'log_pibpc']].copy()
variaveis.columns = ['Pop. (mi)', 'Emp. (mi)', 'Horas (avh)', 'Log PIB/capita']
corr = variaveis.corr()

sns.set_theme(style='white', rc={'figure.figsize': (7, 6)})

g = sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)

g.set_title('Correlação entre variáveis da PWT — 2010', fontsize=13)

A correlação negativa entre horas trabalhadas e log PIB per capita (`-0.50`) confirma o padrão que observamos nos gráficos anteriores.

### Temas e paletas <a id="temas"></a>

Agora que conhecemos os principais tipos de gráfico, podemos explorar a **personalização visual**. O Seaborn oferece cinco temas pré-definidos que controlam o fundo e a grade — basta passar o nome para `sns.set_theme(style=...)`:

| Tema | Descrição |
|------|-----------|
| `'darkgrid'` | Fundo escuro com grade (padrão) |
| `'whitegrid'` | Fundo branco com grade cinza |
| `'dark'` | Fundo escuro, sem grade |
| `'white'` | Fundo branco, sem grade |
| `'ticks'` | Fundo branco com marcações nos eixos |

Além do tema, o argumento `palette=` controla as cores das séries (`'deep'`, `'muted'`, `'Set1'`, `'Set2'`, entre outras). Para comparar os cinco temas de uma vez:


In [ ]:
estilos = ['darkgrid', 'whitegrid', 'dark', 'white', 'ticks']

for estilo in estilos:
    sns.set_theme(style=estilo)

    g = sns.relplot(
        data=df_brazil,
        x='year',
        y='pibpc',
        kind='line',
        marker='o',
        height=4,
        aspect=1.5
    )

    g.set_axis_labels('Ano', 'PIB per capita')
    g.figure.suptitle(f"Tema: '{estilo}'", y=1.03)

O tema escolhido se aplica a **todos** os gráficos seguintes, então é boa prática definir `sns.set_theme(style=...)` no início da análise. Para as próximas seções usaremos `'white'` ou `'whitegrid'`, que combinam bem com publicações.

---


## Plotly: gráficos interativos <a id="plotly"></a>

Até agora todos os gráficos são **estáticos** — imagens fixas. O **Plotly** produz gráficos **interativos** em HTML: é possível passar o mouse sobre os pontos para ver os valores, dar zoom em regiões específicas e clicar na legenda para ocultar ou mostrar séries.

A interface mais simples é o módulo `plotly.express` (importado como `px`), que funciona de forma análoga ao Seaborn: passamos um DataFrame e indicamos as colunas para cada eixo.

> **Nota:** No VS Code, gráficos Plotly são exibidos inline no notebook. Em outros ambientes, pode ser necessário chamar `fig.show()` explicitamente.

### Gráfico de linha interativo <a id="plotly-linha"></a>

Vamos começar reproduzindo o gráfico de horas trabalhadas no Brasil — mas desta vez interativo:

In [ ]:
fig = px.line(
    df_pwtbr,
    x='year',
    y='avh_dia',
    title='Horas médias trabalhadas por dia útil — Brasil',
    labels={'year': 'Ano', 'avh_dia': 'Horas por dia útil'},
    template='simple_white'  # plotly, plotly_white, plotly_dark, ggplot2, seaborn, simple_white, none
)

fig.show()

Passe o mouse sobre a linha para ver o valor exato de cada ano. No canto superior direito há ferramentas de zoom, seleção e download da imagem.

### Múltiplos países <a id="plotly-multi"></a>

Assim como no Seaborn, o argumento `color=` separa automaticamente os grupos — e o Plotly adiciona interatividade na legenda:

In [ ]:
paises_sel = ['Brazil', 'United States', 'Germany', 'Japan', 'China']
df_plotly_multi = df_pwt[df_pwt['country'].isin(paises_sel)].copy()
df_plotly_multi['avh_dia'] = df_plotly_multi['avh'] / 260
df_plotly_multi['country'] = df_plotly_multi['country'].replace({'United States': 'EUA'})

fig = px.line(
    df_plotly_multi,
    x='year',
    y='avh_dia',
    color='country',
    title='Horas médias trabalhadas por dia útil — países selecionados',
    labels={'year': 'Ano', 'avh_dia': 'Horas por dia útil', 'country': 'País'},
    template='simple_white'
)

fig.update_layout(legend_title_text='País')
fig.show()

Clique nos nomes na legenda para ativar ou desativar países. Dê um duplo clique para isolar apenas aquele país.

### Dispersão interativa <a id="plotly-scatter"></a>

O gráfico de dispersão interativo é especialmente útil porque permite **identificar países individualmente** ao passar o mouse. O argumento `hover_name=` define o rótulo principal, e `hover_data=` adiciona campos extras:

In [ ]:
fig = px.scatter(
    df_2010,
    x='avh',
    y='log_pibpc',
    color='grupo',
    color_discrete_map={
        'Alta renda':  '#2ca02c',
        'Renda média': '#ff7f0e',
        'Baixa renda': '#d62728'
    },
    hover_name='country',
    hover_data={'avh': True, 'log_pibpc': ':.2f', 'grupo': False},
    title='Horas trabalhadas vs. PIB per capita — 2010',
    labels={
        'avh':       'Horas médias trabalhadas no ano',
        'log_pibpc': 'Log do PIB per capita',
        'grupo':     'Grupo de renda'
    },
    template='simple_white',
)

fig.show()

Passe o mouse sobre os pontos para ver o nome do país e seus valores exatos. Isso seria impossível em um gráfico estático com 100+ países.

### Gráfico de barras interativo <a id="plotly-bar"></a>

Por fim, um gráfico de barras com os 10 países de maior PIB per capita em 2010, mostrando as horas trabalhadas:

In [ ]:
df_top10_p = df_pwt[df_pwt['year'] == 2010].dropna(subset=['pibpc', 'avh']).copy()
df_top10_p = df_top10_p.sort_values('pibpc', ascending=False).iloc[:10]
df_top10_p['country'] = df_top10_p['country'].replace({
    'United Arab Emirates': 'UAE',
    'United States': 'EUA'
})
df_top10_p = df_top10_p.sort_values('avh', ascending=True)

fig = px.bar(
    df_top10_p,
    x='avh',
    y='country',
    orientation='h',
    color='avh',
    color_continuous_scale='Blues',
    hover_data={'pibpc': ':,.0f', 'avh': True},
    title='Horas trabalhadas — 10 países com maior PIB per capita (2010)',
    labels={
        'avh': 'Horas médias trabalhadas no ano',
        'country': 'País',
        'pibpc': 'PIB per capita (USD 2017)'
    },
    template='simple_white'
)

fig.update_layout(coloraxis_showscale=False) # remove a barra lateral de cores
fig.show()

O potencial para criar gráficos interativos, dinâmicos e informativos é enorme!

In [ ]:
# Retorna dados de países ao longo dos anos.
df = px.data.gapminder()

fig = px.scatter(
    df,
    x="gdpPercap",
    y="lifeExp",
    animation_frame="year",    # Cada ano vira um frame da animação.
    animation_group="country", # Garante que cada país mantenha a mesma cor ao longo da animação.
    size="pop",
    color="continent",
    hover_name="country",
    log_x=True,
    size_max=55,
    range_x=[100, 100000],
    range_y=[25, 90],
    template='simple_white'
)
fig.show()

## Quando usar cada biblioteca? <a id="comparacao"></a>

As três bibliotecas que vimos até agora têm pontos fortes complementares:

| Situação | Recomendação |
|----------|-------------|
| Publicação, artigo, relatório em PDF | **Matplotlib** — controle total sobre cada detalhe visual |
| Análise exploratória, visualizações estatísticas | **Seaborn** — menos código, grupos e distribuições com `hue=` |
| Exploração interativa, apresentações web | **Plotly** — zoom, hover, filtros na legenda |

Na prática, é comum combinar as bibliotecas: usar Seaborn para explorar os dados durante a análise e, depois, reproduzir o gráfico final no Matplotlib com personalização detalhada para publicação — ou no Plotly para apresentações interativas.

## Conclusão

Nesta aula vimos:

- **Seaborn** — funções de nível figura: `relplot` (linhas e dispersão), `displot` (histograma e KDE), `catplot` (boxplot) e `heatmap` — todas com suporte nativo a agrupamentos via `hue=` e personalização via `g.set_axis_labels()`, `g.despine()` etc.
- **Plotly Express** — gráficos interativos com `px.line`, `px.scatter` e `px.bar` — com hover, zoom e filtros automáticos.

**Referências**

- Documentação Seaborn: [seaborn.pydata.org](https://seaborn.pydata.org)
- Documentação Plotly: [plotly.com/python](https://plotly.com/python)
